In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient
import pickle
from glob import glob
from itertools import groupby
from operator import itemgetter

import sys
import os
sys.path.append(os.path.abspath("../../"))
from neuralforecast.losses.numpy import mae, mse, rmse, mape, smape, mase

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("/Users/silas.schroeder@sap.com/qtsf")
client = MlflowClient()

# Define experiment label (should match what was used in model-training.ipynb)
EXPERIMENT = "dev"

In [ ]:
# Load test data
df = pd.read_csv("../data/train_prepared.csv", parse_dates=["Date"], low_memory=False)
df = df.sort_values(["Store", "Date"])

delta_days = (max(df["Date"]) - min(df["Date"])).days
train_days = int(delta_days * 0.8)
split_date = min(df["Date"]) + np.timedelta64(train_days,"D")

train_df = df[df["Date"] < split_date].copy()
test_df  = df[df["Date"] >= split_date].copy()

HORIZON = (max(df["Date"]) - np.datetime64(split_date)).days
y_train_global = train_df["Sales"].values
SEASONALITY = 7

print(f"Test period: {test_df['Date'].min().date()} to {test_df['Date'].max().date()}")
print(f"Horizon: {HORIZON} days")

In [ ]:
# Retrieve all runs from MLflow for this experiment
experiment = client.get_experiment_by_name("/Users/silas.schroeder@sap.com/qtsf")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.experiment = '{EXPERIMENT}'",
    order_by=["start_time DESC"]
)

print(f"Found {len(runs)} runs for experiment '{EXPERIMENT}'")

# Group runs by run_label and model
run_data = {}
for run in runs:
    run_label = run.data.params.get("run_label")
    model = run.data.params.get("model")
    
    if run_label and model:
        if run_label not in run_data:
            run_data[run_label] = {}
        
        run_data[run_label][model] = {
            "metrics": run.data.metrics,
            "params": run.data.params,
            "run_id": run.info.run_id
        }

print(f"\nRun labels found: {list(run_data.keys())}")
for run_label, models in run_data.items():
    print(f"  {run_label}: {list(models.keys())}")

In [ ]:
# Reconstruct all_avg_results and all_seed_results from MLflow
all_avg_results = {}
all_seed_results = {}
all_param_counts = {}
all_training_times = {}

for run_label, models in run_data.items():
    all_avg_results[run_label] = {}
    all_seed_results[run_label] = {}
    all_training_times[run_label] = []
    all_param_counts[run_label] = {}
    
    for model, data in models.items():
        metrics = data["metrics"]
        
        # Extract avg metrics (these are the seed-ensemble metrics)
        all_avg_results[run_label][model] = {
            "MAE": metrics.get("avg_MAE", 0),
            "MSE": metrics.get("avg_MSE", 0),
            "RMSE": metrics.get("avg_RMSE", 0),
            "MAPE": metrics.get("avg_MAPE", 0),
            "sMAPE": metrics.get("avg_sMAPE", 0),
            "MASE": metrics.get("avg_MASE", 0),
        }
        
        # Extract seed statistics
        all_seed_results[run_label][model] = {}
        for metric_name in ["MAE", "MSE", "RMSE", "MAPE", "sMAPE", "MASE"]:
            mean_val = metrics.get(f"mean_{metric_name}", 0)
            std_val = metrics.get(f"std_{metric_name}", 0)
            # Reconstruct approximate per-seed values (for visualization)
            # Note: This is an approximation. For exact values, we'd need to log them separately
            all_seed_results[run_label][model][metric_name] = [mean_val] * 3  # assuming 3 seeds
        
        # Extract parameter counts
        param_count = metrics.get("num_parameters")
        if param_count is not None:
            all_param_counts[run_label][model] = int(param_count)
        
        # Extract training times
        mean_time = metrics.get("mean_training_time_s")
        if mean_time is not None and mean_time not in all_training_times[run_label]:
            all_training_times[run_label].append(mean_time)

print("\nReconstructed data structures from MLflow:")
print(f"Run labels: {list(all_avg_results.keys())}")
print(f"Example metrics for first model in first run:")
first_run = list(all_avg_results.keys())[0]
first_model = list(all_avg_results[first_run].keys())[0]
print(f"  {first_run}/{first_model}: {all_avg_results[first_run][first_model]}")

In [ ]:
forecast_dir = 'forecasts'
all_avg_forecasts = {}
all_seed_forecasts = {}

if os.path.exists(forecast_dir):
    # Finde alle forecasts.pkl in Unterordnern (classical, naive, ideal, noisy_sim, etc.)
    forecast_files = glob(os.path.join(forecast_dir, '*', 'forecasts.pkl'))
    
    if forecast_files:
        print(f"✓ Found {len(forecast_files)} forecast files:")
        
        for fpath in forecast_files:
            # Extrahiere run_label aus dem Ordnernamen
            run_label = os.path.basename(os.path.dirname(fpath))
            
            with open(fpath, 'rb') as f:
                forecast_data = pickle.load(f)
            
            # forecast_data kann verschiedene Strukturen haben:
            # 1. Dict mit 'all_avg_forecasts' und 'all_seed_forecasts' keys
            # 2. Direkt ein DataFrame mit Forecasts
            if isinstance(forecast_data, dict):
                if 'all_avg_forecasts' in forecast_data:
                    # Nested structure - extrahiere für dieses run_label
                    all_avg_forecasts[run_label] = forecast_data['all_avg_forecasts'].get(run_label, forecast_data['all_avg_forecasts'])
                    if 'all_seed_forecasts' in forecast_data:
                        all_seed_forecasts[run_label] = forecast_data['all_seed_forecasts'].get(run_label, forecast_data['all_seed_forecasts'])
                else:
                    # Dict ist direkt das forecast DataFrame oder enthält model columns
                    all_avg_forecasts[run_label] = forecast_data
            else:
                # DataFrame direkt
                all_avg_forecasts[run_label] = forecast_data
            
            print(f"  - {run_label}: {fpath}")
            if isinstance(all_avg_forecasts[run_label], pd.DataFrame):
                print(f"    Columns: {list(all_avg_forecasts[run_label].columns)}")
        
        print(f"\n✓ Loaded forecasts for run labels: {list(all_avg_forecasts.keys())}")
        print("  Store-level plots are now available!")
    else:
        print(f"⚠️  Warning: No forecasts.pkl files found in {forecast_dir}/*/")
        print("   Run model-training.ipynb first to generate forecasts.")
else:
    print(f"⚠️  Warning: {forecast_dir}/ directory not found.")
    print("   Run model-training.ipynb first to generate forecasts.")
    print("   Store-level plotting cells will be skipped.")

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print("\nTest-set metrics  (seed-ensemble ★ | seeds mean ± std)")
print("─" * 70)
for run_label in all_avg_results:
    print(f"\n  [{run_label}]")
    for model in sorted(all_avg_results[run_label], key=lambda m: all_avg_results[run_label][m]["sMAPE"]):
        a = all_avg_results[run_label][model]
        s = all_seed_results[run_label][model]
        print(f"    {model}:")
        for metric in ["MAE", "MSE", "RMSE", "MAPE", "sMAPE", "MASE"]:
            print(f"      {metric}: ★{a[metric]:.4f}  |  "
                  f"seeds {np.mean(s[metric]):.4f} ± {np.std(s[metric]):.4f}")

In [ ]:
# ── sMAPE comparison: Classical vs Quantum (Ideal & Noisy) ───────────────────
# Eine Grafik mit 4 Spalten (DLinear, NHITS, TimesNet, DeepAR)
# Pro Spalte: Classical, Ideal, Noisy als Punkte übereinander
# Horizontale Linie für Naive Baseline

fig, ax = plt.subplots(figsize=(10, 6))

# Modelle und ihre Spalten
model_bases = ["DLinear", "NHITS", "TimesNet", "DeepAR"]
x_positions = np.arange(len(model_bases))

# Farben
color_classical = "#1f77b4"  # Blau
color_ideal = "#2ca02c"      # Grün
color_noisy = "#d62728"      # Rot
color_naive = "#7f7f7f"      # Grau

# Naive Baseline (horizontale Linie)
if "naive" in all_avg_results and "Naive" in all_avg_results["naive"]:
    naive_smape = all_avg_results["naive"]["Naive"].get("sMAPE", None)
    if naive_smape:
        ax.axhline(y=naive_smape, color=color_naive, linestyle="--", linewidth=2, label=f"Naive (sMAPE {naive_smape:.2f})")

# Daten sammeln und plotten
# Reihenfolge: Quadrate (hinten) -> Kreise (mitte) -> Dreiecke (vorne)
for i, base_model in enumerate(model_bases):
    x = x_positions[i]
    quantum_model = f"Q{base_model}"
    
    # 1. Quantum Noisy (Quadrate - hinten, zorder=3)
    if "noisy_sim" in all_avg_results and quantum_model in all_avg_results["noisy_sim"]:
        smape_val = all_avg_results["noisy_sim"][quantum_model].get("sMAPE", None)
        if smape_val:
            ax.scatter(x, smape_val, color=color_noisy, s=200, zorder=3, marker="s",
                      label="Quantum (Noisy)" if i == 0 else "")
            ax.text(x + 0.15, smape_val, f"{smape_val:.2f}", va='center', fontsize=9)
    
    # 2. Classical (Kreise - mitte, zorder=4)
    if "classical" in all_avg_results and base_model in all_avg_results["classical"]:
        smape_val = all_avg_results["classical"][base_model].get("sMAPE", None)
        if smape_val:
            ax.scatter(x, smape_val, color=color_classical, s=150, zorder=4, 
                      label="Classical" if i == 0 else "")
            ax.text(x + 0.15, smape_val, f"{smape_val:.2f}", va='center', fontsize=9)
    
    # 3. Quantum Ideal (Dreiecke - vorne, zorder=5)
    if "ideal" in all_avg_results and quantum_model in all_avg_results["ideal"]:
        smape_val = all_avg_results["ideal"][quantum_model].get("sMAPE", None)
        if smape_val:
            ax.scatter(x, smape_val, color=color_ideal, s=120, zorder=5, marker="^",
                      label="Quantum (Ideal)" if i == 0 else "")
            ax.text(x + 0.15, smape_val, f"{smape_val:.2f}", va='center', fontsize=9)

# Achsen und Labels
ax.set_xticks(x_positions)
ax.set_xticklabels(model_bases, fontsize=11)
ax.set_xlabel("Model Architecture", fontsize=11)
ax.set_ylabel("sMAPE (seed-ensemble)", fontsize=11)
ax.set_title("sMAPE Comparison: Classical vs Quantum Models", fontsize=13, pad=40)
ax.grid(True, axis="y", alpha=0.3)

# Legende zwischen Überschrift und Bildfläche (oben)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=4, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── sMAPE comparison bar chart across all runs ─────────────────────────────────

run_labels = [rl for rl in all_avg_results.keys() if rl != "naive"]  # Exclude naive
n_runs = len(run_labels)

fig, ax = plt.subplots(figsize=(max(12, 3 * n_runs), 5))

# Für jeden run_label die vorhandenen Modelle sammeln
bar_data = []
for rl in run_labels:
    models = sorted(all_avg_results[rl].keys())
    for model in models:
        smape_val = all_avg_results[rl][model].get("sMAPE", float("nan"))
        bar_data.append((rl, model, smape_val))

# Berechne Positionen
x_pos = []
labels = []
colors = plt.cm.Set2(np.linspace(0, 1, 8))  # Farbpalette
bar_colors = []

current_x = 0
group_centers = []
group_labels = []

for rl in run_labels:
    models_in_run = [(m, s) for (r, m, s) in bar_data if r == rl]
    group_start = current_x
    for i, (model, smape_val) in enumerate(models_in_run):
        x_pos.append(current_x)
        labels.append(model)
        # Farbe basierend auf Modellname (konsistent über run_labels)
        model_idx = hash(model) % len(colors)
        bar_colors.append(colors[model_idx])
        current_x += 1
    group_centers.append((group_start + current_x - 1) / 2)
    group_labels.append(rl)
    current_x += 1.5  # Lücke zwischen Gruppen

# Plot
smape_values = [s for (_, _, s) in bar_data]
bars = ax.bar(x_pos, smape_values, width=0.8, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)

# Berechne Y-Offset relativ zur Datenskala (1% des Maximalwerts)
y_offset = max(smape_values) * 0.01

# Labels knapp oberhalb der Balken
for i, (pos, val, label) in enumerate(zip(x_pos, smape_values, labels)):
    ax.text(pos, val + y_offset, label, ha='center', va='bottom', fontsize=8, rotation=45)

# X-Achse: Gruppenlabels
ax.set_xticks(group_centers)
ax.set_xticklabels(group_labels, fontsize=11)

ax.set_ylabel("sMAPE (seed-ensemble)")
ax.set_title("sMAPE comparison: classical baseline vs. quantum devices")
ax.grid(True, axis="y", alpha=0.3)

# Legende für Modelltypen (einmalig)
unique_models = sorted(set(labels))
legend_handles = []
for model in unique_models:
    model_idx = hash(model) % len(colors)
    legend_handles.append(plt.Rectangle((0,0), 1, 1, fc=colors[model_idx], alpha=0.85, label=model))
ax.legend(handles=legend_handles, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Check if forecast data is available for store-level plots
if not all_avg_forecasts:
    print("⚠️  Skipping store-level plots: forecast data not available.")
    print("   Run model-training.ipynb and ensure forecasts.pkl is created.")
else:
    # ── Helper: compute sMAPE for two arrays ──
    def compute_smape(actual, predicted):
        """Compute sMAPE between actual and predicted values."""
        actual = np.array(actual)
        predicted = np.array(predicted)
        denominator = (np.abs(actual) + np.abs(predicted)) / 2
        # Avoid division by zero
        mask = denominator > 0
        if mask.sum() == 0:
            return 0.0
        return np.mean(np.abs(actual[mask] - predicted[mask]) / denominator[mask]) * 100

    # ── Helper: compute per-store sMAPE for a specific model across run labels ──
    def store_smape_for_model(model_name, run_labels_to_use):
        """Returns a Series indexed by Store with mean sMAPE across specified run labels."""
        per_store = []

        for run_label in run_labels_to_use:
            if run_label not in all_avg_forecasts:
                continue
            fc = all_avg_forecasts[run_label]
            # Determine column name (classical uses model_name, quantum uses Q+model_name)
            if run_label == "classical":
                col = model_name
            else:
                col = f"Q{model_name}"
            
            if col not in fc.columns:
                continue
                
            comp = test_df[["Store", "Date", "Sales"]].merge(fc, on=["Store", "Date"], how="inner")
            store_err = comp.groupby("Store").apply(
                lambda g: compute_smape(g["Sales"].values, g[col].values), include_groups=False
            )
            per_store.append(store_err)

        if not per_store:
            return pd.Series(dtype=float)
        return pd.concat(per_store, axis=1).mean(axis=1)


    # ── Helper: plot one store comparing classical vs quantum variants ──
    def plot_store_model_comparison(ax, store_id, model_name, title):
        """Plot actual + classical + ideal + noisy_sim for a single model."""
        store_actual = test_df[test_df["Store"] == store_id].sort_values("Date")

        # Plot actual
        ax.plot(store_actual["Date"], store_actual["Sales"],
                label="Actual", color="black", linewidth=2)

        # Plot classical model
        if "classical" in all_avg_forecasts:
            classical_fc = all_avg_forecasts["classical"]
            if model_name in classical_fc.columns:
                store_classical = classical_fc[classical_fc["Store"] == store_id].sort_values("Date")
                if not store_classical.empty:
                    actual_vals = store_actual.set_index("Date")["Sales"].reindex(store_classical["Date"].values).values
                    classical_smape = compute_smape(actual_vals, store_classical[model_name].values)
                    ax.plot(store_classical["Date"], store_classical[model_name],
                            label=f"{model_name} (sMAPE {classical_smape:.1f}%)",
                            color="blue", linestyle="-", linewidth=1.5)

        # Plot ideal quantum model
        quantum_model = f"Q{model_name}"
        if "ideal" in all_avg_forecasts:
            ideal_fc = all_avg_forecasts["ideal"]
            if quantum_model in ideal_fc.columns:
                store_ideal = ideal_fc[ideal_fc["Store"] == store_id].sort_values("Date")
                if not store_ideal.empty:
                    actual_vals = store_actual.set_index("Date")["Sales"].reindex(store_ideal["Date"].values).values
                    ideal_smape = compute_smape(actual_vals, store_ideal[quantum_model].values)
                    ax.plot(store_ideal["Date"], store_ideal[quantum_model],
                            label=f"{quantum_model} ideal (sMAPE {ideal_smape:.1f}%)",
                            color="green", linestyle="-", linewidth=1.5)

        # Plot noisy_sim quantum model
        if "noisy_sim" in all_avg_forecasts:
            noisy_fc = all_avg_forecasts["noisy_sim"]
            if quantum_model in noisy_fc.columns:
                store_noisy = noisy_fc[noisy_fc["Store"] == store_id].sort_values("Date")
                if not store_noisy.empty:
                    actual_vals = store_actual.set_index("Date")["Sales"].reindex(store_noisy["Date"].values).values
                    noisy_smape = compute_smape(actual_vals, store_noisy[quantum_model].values)
                    ax.plot(store_noisy["Date"], store_noisy[quantum_model],
                            label=f"{quantum_model} noisy (sMAPE {noisy_smape:.1f}%)",
                            color="red", linestyle="--", linewidth=1.5)

        ax.set_title(title)
        ax.legend(fontsize=9, loc="upper left")
        ax.grid(True, alpha=0.3)
        ax.set_ylabel("Sales")


    # ── Define models to compare (classical name only) ────────────────────────────
    models_to_compare = ["DLinear", "NHITS", "TimesNet", "DeepAR"]
    
    # Filter to only models that exist in forecasts
    available_models = []
    for model in models_to_compare:
        if "classical" in all_avg_forecasts and model in all_avg_forecasts["classical"].columns:
            available_models.append(model)
    
    if not available_models:
        print("⚠️  No matching models found in forecasts.")
    else:
        # Run labels to use for sMAPE calculation (excluding naive)
        run_labels_for_smape = [rl for rl in all_avg_forecasts.keys() if rl != "naive"]

        # ── Find best/worst stores for each model ───────────────────────────────
        print("Finding best/worst stores for each model (based on sMAPE):")
        model_stores = []
        for model_name in available_models:
            store_smape = store_smape_for_model(model_name, run_labels_for_smape)
            if store_smape.empty:
                print(f"  {model_name} — No data available")
                model_stores.append((None, None))
                continue
            best_store = store_smape.idxmin()
            worst_store = store_smape.idxmax()
            model_stores.append((best_store, worst_store))
            print(f"  {model_name} — "
                  f"best: {best_store} (sMAPE {store_smape[best_store]:.1f}%), "
                  f"worst: {worst_store} (sMAPE {store_smape[worst_store]:.1f}%)")

        # ── Build figure: models × 2 (best/worst) ───────────────────────────────
        n_models = len(available_models)
        fig, axes = plt.subplots(n_models, 2, figsize=(20, 6*n_models), squeeze=False)
        fig.suptitle("Model Comparisons: Classical vs Quantum (Ideal & Noisy)\n"
                     "Best & Worst Store Forecasts (sMAPE)",
                     fontsize=14)

        for idx, (model_name, (best_store, worst_store)) in enumerate(zip(available_models, model_stores)):
            if best_store is None:
                axes[idx][0].text(0.5, 0.5, f"No data for {model_name}", ha='center', va='center')
                axes[idx][1].text(0.5, 0.5, f"No data for {model_name}", ha='center', va='center')
                continue
                
            # Best store
            plot_store_model_comparison(axes[idx][0], best_store, model_name,
                                       f"{model_name} — Best Store {best_store}")

            # Worst store
            plot_store_model_comparison(axes[idx][1], worst_store, model_name,
                                       f"{model_name} — Worst Store {worst_store}")

        plt.tight_layout()
        plt.show()